# Measuring Your Heart Rate with a Smartphone Camera

### A self-contained Colab on *contact photoplethysmography (PPG)*

**ACM Summer School on AI for Health — Hands-on Notebook**  
*Prepared by Nipun Batra's group*

---

When you press your fingertip lightly over a phone's **rear camera with the flash
(torch) turned ON**, every heartbeat pushes a pulse of blood through the
capillaries in your finger. More blood ⇒ more light absorbed ⇒ the image gets
*slightly darker*; between beats it gets *slightly brighter*. The camera is, in
effect, an optical blood-volume sensor.

This tiny brightness oscillation is a **photoplethysmogram (PPG)** — the same
signal a pulse-oximeter or a smartwatch measures. Its frequency is your heart
rate.

In this notebook you will:

1. **Build intuition** by synthesising a clean PPG signal at a known heart rate and recovering it — so you can trust the pipeline before trusting real data.
2. **Process a real 20 s finger video** recorded with a phone camera and read off the heart rate.

> ⚠️ **This is an educational demo, not a medical device.** Do not use it for any diagnostic or clinical decision.


## 0. The Physics in One Paragraph

Light from the LED flash enters the finger, scatters through tissue, and a
fraction returns to the camera sensor. Oxygenated blood (haemoglobin) absorbs
this light. With each cardiac cycle the arterial blood volume rises (systole)
and falls (diastole), so the amount of returning light is **amplitude-modulated
at the heart rate** (≈ 0.7–3 Hz, i.e. ~42–180 bpm). The camera already averages
millions of photons over the whole frame, so simply taking the **mean pixel
intensity of each frame** gives us a 1-D time series that *is* the PPG. The
**red channel** carries the strongest signal under a white/torch LED because
red light penetrates tissue best.

The whole algorithm is therefore:

```
video frames ──▶ per-frame mean(red channel) ──▶ 1-D signal s(t)
   s(t) ──▶ detrend ──▶ band-pass 0.7–4 Hz ──▶ FFT / peak-detect ──▶ heart rate
```


## 1. Setup

Clones the repo so the finger video is available at a known path, then imports dependencies.


In [ ]:
import os, subprocess

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    subprocess.run(["git", "clone", "https://github.com/parvthacker/ai-health-sensing.git"])
    os.chdir("ai-health-sensing")
    print(" Repo cloned")
else:
    print("Running locally — skipping clone")


In [ ]:
try:
    import cv2
except ImportError:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless"], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks, welch, detrend

np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})
print(" Imports done")


## 2. The Signal-Processing Pipeline

These four functions are the entire heart-rate estimator. We define them up
front and apply the *exact same* functions to (a) a synthetic signal and
(b) the real finger video — so there is no hidden magic.


In [ ]:
def bandpass(x, fs, lo=0.7, hi=4.0, order=3):
    """Zero-phase Butterworth band-pass. Keeps plausible heart-rate frequencies
    (lo..hi Hz) and removes slow baseline drift + high-frequency noise."""
    nyq = 0.5 * fs
    hi = min(hi, 0.99 * nyq)              # guard against fs too low for hi
    b, a = butter(order, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a, x)


def hr_from_fft(x, fs, lo=0.7, hi=4.0):
    """Heart rate (bpm) = frequency of the largest spectral peak in [lo, hi]."""
    x = detrend(np.asarray(x, float))
    n = len(x)
    win = np.hanning(n)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    power = np.abs(np.fft.rfft(x * win)) ** 2
    band = (freqs >= lo) & (freqs <= hi)
    f_peak = freqs[band][np.argmax(power[band])]
    return f_peak * 60.0, freqs, power


def hr_from_peaks(x, fs, lo=0.7, hi=4.0):
    """Heart rate from time-domain beat-to-beat (peak) detection.
    Also returns the peak indices and inter-beat intervals (for HRV)."""
    xf = bandpass(detrend(np.asarray(x, float)), fs, lo, hi)
    min_dist = int(fs / hi)              # beats can't be closer than 1/hi seconds
    peaks, _ = find_peaks(xf, distance=max(1, min_dist),
                          prominence=0.3 * np.std(xf))
    ibi = np.diff(peaks) / fs            # seconds between successive beats
    hr = 60.0 / np.median(ibi) if len(ibi) else np.nan
    return hr, peaks, xf, ibi


def signal_quality(x, fs, lo=0.7, hi=4.0):
    """A crude SNR: power in the heart-rate band vs. total power.
    Values > ~0.2 usually mean a usable recording."""
    f, P = welch(detrend(np.asarray(x, float)), fs=fs,
                 nperseg=min(len(x), int(fs * 8)))
    band = (f >= lo) & (f <= hi)
    return P[band].sum() / P.sum()

## 3. From Video to Signal

This is the only piece that touches video. `extract_ppg_from_video` opens a
file, reads it frame-by-frame, and records the **mean of one colour channel**
per frame. Note OpenCV stores frames as **BGR**, not RGB.


In [ ]:
import cv2

def extract_ppg_from_video(path, channel="red", roi_frac=0.6, max_seconds=None):
    """Return (signal, fps). `signal[i]` = mean of the chosen channel over a
    central ROI of frame i. A central ROI avoids dark vignette corners."""
    ch_idx = {"blue": 0, "green": 1, "red": 2}[channel]   # BGR order
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    max_frames = int(max_seconds * fps) if max_seconds else None
    means = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        ch, cw = int(h * roi_frac), int(w * roi_frac)
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        roi = frame[y0:y0 + ch, x0:x0 + cw, ch_idx]
        means.append(float(roi.mean()))
        if max_frames and len(means) >= max_frames:
            break
    cap.release()
    return np.asarray(means), float(fps)

## 4. Real Finger Video — Heart Rate

The video was recorded by pressing a finger pad over the **rear camera and flash**
of a smartphone. The flash stays on throughout; the camera sees nothing but a
uniformly lit red field — the brightness oscillation from blood pulsing through
the capillaries is invisible to the eye but clear in the signal.

**How to record your own clip:**
1. Rear camera, flash/torch confirmed ON before and during recording
2. Finger pad covering both lens and flash completely
3. Light contact — rest, don't press
4. Hand resting on a table, 20–30 seconds


In [ ]:
video_path = "phone_ppg/finger_video.mp4"

# Try red channel first; fall back to green if quality is poor
sig, fps = extract_ppg_from_video(video_path, channel="red", roi_frac=0.6)
q_red = signal_quality(sig, fps)
if q_red < 0.15:
    sig_g, _ = extract_ppg_from_video(video_path, channel="green", roi_frac=0.6)
    if signal_quality(sig_g, fps) > q_red:
        print("Red channel weak — switching to green.")
        sig = sig_g

# Drop first 2 s (auto-exposure settling)
skip = int(fps * 2)
sig = sig[skip:] if len(sig) > 3 * skip else sig

hr_fft, freqs, power = hr_from_fft(sig, fps)
hr_pk, peaks, sig_f, ibi = hr_from_peaks(sig, fps)
quality = signal_quality(sig, fps)

print(f"Recording length : {len(sig)/fps:.1f} s @ {fps:.1f} fps")
print(f"Signal quality   : {quality:.2f}   ({'good' if quality > 0.2 else 'weak — try holding more still'})")
print(f"❤️  Heart rate (FFT)  : {hr_fft:.0f} bpm")
print(f"❤️  Heart rate (peaks): {hr_pk:.0f} bpm")


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8))
tt = np.arange(len(sig)) / fps

ax[0].plot(tt, sig, lw=0.9, color="#888")
ax[0].set(title="Raw per-frame mean intensity (your PPG)",
          xlabel="time (s)", ylabel="intensity")

ax[1].plot(tt, sig_f, lw=1.1, color="#d62728")
ax[1].plot(tt[peaks], sig_f[peaks], "kv", ms=6, label="beats")
ax[1].set(title=f"Filtered PPG + beats  →  {hr_pk:.0f} bpm",
          xlabel="time (s)", ylabel="a.u."); ax[1].legend(loc="upper right")

bpm_axis = freqs * 60
m = (bpm_axis >= 30) & (bpm_axis <= 200)
ax[2].plot(bpm_axis[m], power[m] / power[m].max(), color="#2ca02c")
ax[2].axvline(hr_fft, color="r", ls="--", label=f"peak = {hr_fft:.0f} bpm")
ax[2].set(title="Spectrum", xlabel="heart rate (bpm)", ylabel="norm. power")
ax[2].legend()
plt.tight_layout(); plt.show()

## 5. Bonus: Heart-Rate Variability (HRV)

The spacing between successive beats is not perfectly constant — it speeds up
and slows down (partly with your breathing). The standard deviation of those
inter-beat intervals is a simple HRV measure. With a phone camera it's only a
rough estimate (frame rate limits timing resolution), but it illustrates the concept.


In [ ]:
if len(ibi) > 3:
    ibi_ms = ibi * 1000.0
    sdnn = np.std(ibi_ms)
    print(f"Mean inter-beat interval : {ibi_ms.mean():.0f} ms")
    print(f"SDNN (HRV proxy)         : {sdnn:.0f} ms")
    plt.figure(figsize=(11, 3))
    plt.plot(ibi_ms, "o-", color="#9467bd")
    plt.title("Inter-beat intervals"); plt.xlabel("beat #")
    plt.ylabel("interval (ms)"); plt.tight_layout(); plt.show()
else:
    print("Not enough beats detected for HRV — record a longer, steadier clip.")


# Why this works, where it fails, and things to try

**Why it works.** The flash provides a stable light source; the finger is a
fixed, well-perfused tissue bed; averaging the whole frame gives a high-SNR
optical pulse. This *contact* PPG is far easier than *remote* PPG (rPPG) from a
face video, where the signal is ~100× weaker and motion/lighting dominate.

**Common failure modes**
- *Torch off* → no modulation, just noise. The single most common mistake.
- *Pressing too hard* → occludes capillaries, flattens the pulse.
- *Motion* → broadband noise; the `signal_quality` score drops.
- *Low/again variable frame rate* → wrong time axis. We read `fps` from the
  file; if a phone reports a wrong fps, the bpm will be scaled by the same
  factor. Cross-check against counting your pulse for 15 s × 4.

**Exercises**
1. Compare red vs. green vs. blue channels — which gives the cleanest spectrum,
   and why? (Hint: optical penetration depth vs. haemoglobin absorption.)
2. Replace the FFT peak-pick with **parabolic interpolation** around the peak
   for sub-bin frequency resolution. How much does the estimate change?
3. Add a **sliding window** (e.g. 8 s, 1 s hop) to plot heart rate *over time*
   and watch it change if you hold your breath or after light exercise.
4. Try the **chrominance / POS** algorithms used for face-based rPPG and see
   why contact PPG is so much easier.

**Safety.** Educational only. Not validated, not a medical device. Don't make
health decisions from this.

---
*Made for the ACM Summer School on AI for Health.*